# HW_05 · Corridas con GPU en Colab

Este equipo no tiene GPU NVIDIA, así que aquí se corren:

1. Las combinaciones `cuda` del grid de entrevistas (small y medium, float16 e int8_float16).
2. La transcripción de las llamadas en cada nivel L0–L3 con la configuración por defecto (medium / cuda / float16).

Antes de empezar: **Entorno de ejecución → Cambiar tipo de entorno de ejecución → T4 GPU**.

Necesitas el archivo `audios_colab.zip` (está en `data/audio/_colab/` del repo local). Los audios no se suben a GitHub.

In [ ]:
# 1. Comprobar que hay GPU
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# 2. Clonar el repo e instalar las dependencias fijadas
!git clone -q -b feat/ejecucion https://github.com/leruizdelavegat-max/HW_05_202602.git
%cd HW_05_202602
!pip install -q -r requirements.txt

# CTranslate2 (faster-whisper) se compila contra CUDA 12 / cuDNN 9; el Colab actual trae CUDA 13
# y falla con "libcublas.so.12 is not found". Se instalan esas librerías y se exponen a los !python.
!pip install -q nvidia-cublas-cu12 "nvidia-cudnn-cu12==9.*"
import os, nvidia.cublas.lib, nvidia.cudnn.lib
os.environ["LD_LIBRARY_PATH"] = ":".join(
    [nvidia.cublas.lib.__path__[0], nvidia.cudnn.lib.__path__[0], os.environ.get("LD_LIBRARY_PATH", "")]
)

In [ ]:
# 3. Subir los audios (audios_colab.zip) y descomprimirlos en data/audio/
from google.colab import files

uploaded = files.upload()
!unzip -oq audios_colab.zip -d data/audio
!ls data/audio/entrevistas data/audio/llamadas

In [ ]:
# 4. Escalera de degradación L0-L3 de las llamadas (determinista: semilla 42)
!python scripts/degrade.py

In [ ]:
# 5. Entrevistas: solo las combinaciones GPU (las de CPU ya se corrieron en local)
!python scripts/transcribe.py --set entrevistas --device cuda

In [ ]:
# 6. Llamadas: cada nivel L0-L3 con medium / cuda / float16
!python scripts/transcribe.py --set llamadas

In [ ]:
# 7. Prueba de uso de GPU: lo que reportó el runtime en cada transcripción
import json, pathlib

for p in sorted(pathlib.Path('cache/transcripts').rglob('*cuda*.json'))[:5]:
    d = json.loads(p.read_text(encoding='utf-8'))
    print(p.name, '| runtime:', d['runtime_device'], '| GPU:', d['gpu_name'], '| VRAM pico (MiB):', d['vram_peak_mib'])

In [ ]:
# 8. Descargar las transcripciones y resultados para copiarlos al repo local
!zip -qr resultados_gpu.zip cache/transcripts results
files.download('resultados_gpu.zip')